In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# RLLM-9: Sample, filter, imitate: self-training

## RFT Alternative methods, part 2 of the RLLM series


## Where this session fits

> **RLLM-8's first map has a named slot, RLLM-9: the model's own samples, picked by a reward but trained with an SFT loss. This session asks: can a reward pick the data and an SFT loss train on it?**

- **Part 2: RFT Alternative methods**: alternatives to the PPO and GRPO you know, each placed by three axes
- **Axes**: 1 whose samples (RLLM-7), 2 who supplies the weights (a reference or a reward) and 3 how fine (RLLM-8). Here: two new methods
- **Position**: session 9 of 14, after RLLM-8 (Who supplies the weights? A reference or a reward)
- **Goal of Part 2**: a **map** that places any method by the three axes

<img src="images/series_map_9.png" width=55%>


## This session: RLLM-9

- **Question**: can a **reward pick the data** while an **SFT loss trains on it**, and in what sense is that RL?
- **By the end you can**:
  - describe **sample, filter, imitate**: a **filter** (a 0/1 weight chosen by a reward) keeps some completions, SFT trains on them
  - say why a rejected completion gets **weight 0**, not a negative weight
  - say **in what sense**, and for which filter, the loop equals a policy gradient (in expectation, one step)
  - place **self-distillation** (the filter with one constant bar) and **RAFT** (keep the best of $G$ completions per prompt) on the map
  - say what a **hybrid** is (a reward picks the data, a reference trains on it)


## Map: the loop, its two filters, and where they sit

> **One loop and two filters. Sample, filter, imitate is the loop. A constant bar and best-of-$G$ are its two filters. The last section puts the loop in RLLM-8's grid.**

<img src="images/rllm9_map.png" alt="the loop sample, filter, imitate, repeat; under the filter the two filters, self-distillation and RAFT; then the comparison and the placement sections" width=54%>

- This is the **plan**; each box names the section that covers it
- **Loop**: what the loop is, and in what sense it is policy gradient
- **Self-distillation, RAFT, Comparison**: the two filters, and what each keeps and drops
- **Placement**: the hybrid box in RLLM-8's grid. *Next: the loop itself*


## Loop: sample, filter, imitate

> **Sample, filter, imitate: the model samples, a reward decides which completions to keep, and SFT trains on the kept ones. The filter is a 0/1 weight chosen by a reward.**

<img src="images/sample_filter_imitate_loop.png" alt="sample, reward, filter, imitate, repeat; rejected completions get weight 0" width=70%>

- A **bar** is a reward threshold: a completion whose reward clears it is kept
  - a **constant-threshold filter** uses one bar for every prompt
- A kept completion gets **weight 1** on every token. A rejected one gets **weight 0**: no gradient of its own
  - there is no *explicit* push down on a rejected completion. It is lowered only indirectly, by normalization (the probabilities sum to 1)
- The parameters change only in the SFT step. The reward enters through the filter
- *Next: in what sense is this reinforcement learning?*


## Loop: a filter makes this loop a policy-gradient step

> **For a constant-threshold filter, sample, filter, imitate is, in expectation and for one step, a policy-gradient step on "the probability that a completion clears the bar".**

<img src="images/filter_expected_step.png" alt="toy policy: with equal weights the exact average push is zero; with a 0/1 filter weight the exact average push raises the actions that clear the bar" width=52%>

<!-- seed-pay: zero-push-equal-weights -->
- From RLLM-8: equal weights on own completions push nowhere on average (left)
- The filter weights each completion by $R(\y)$: 1 if its reward clears the bar, else 0
  - $\y$ is a completion sampled from the current policy for a prompt, and the objective is $J=\Exp{}[R(\y)]$
  - a **weight chosen by a reward**, so the push is no longer zero (right)
- Recall RLLM-5: a policy-gradient step raises the probability of what earned reward. The filter makes this loop do that
- **The kind of claim**: an equivalence of **updates**, one expected step
  - it says nothing about where training ends, and no KL penalty (RLLM-2) holds it near the start
  - several SFT steps on one batch go stale: RLLM-7, "How stale is too stale?"
  - derivation and conditions: skipped asides "why a constant-threshold filter is a policy-gradient step" and "conditions on the equivalence"
- *Next: the first filter, one constant bar*


## Self-distillation: a filter with one constant threshold

> **Self-distillation, as we use the word: the loop whose filter keeps every completion whose reward clears one constant threshold, the same for every prompt.**

- **Why this section?** The punchline needs a filter. There are two common ones, and they keep different completions
- *Names: this is not self-critique (RLLM-3); see the skipped aside "names that clash"*

```python
policy = sft_checkpoint
for rnd in range(N):
    kept = []
    for prompt in prompts:                     # every training prompt
        completions = sample(policy, prompt, G, temperature=1.0)   # G per prompt; 1.0 = the model's own probabilities
        for c in completions:
            if reward(c) >= threshold:         # one constant, same for every prompt
                kept.append((prompt, c))
    policy = sft(policy, kept)                 # imitate the keepers: several steps on this one batch
```

- **Easy prompts** contribute many completions, **hard prompts** few or none, so the kept set changes size from round to round
- Too high a bar keeps nothing. Too low keeps everything, and we are back to equal weights, with no push
- *Next: the second filter, a rank within the prompt*


## RAFT: the same loop, a different filter

> **RAFT (reward-ranked fine-tuning, Dong et al., 2023) keeps the best of the $G$ completions of each prompt, so every prompt contributes exactly one.**

```python
policy = sft_checkpoint
for rnd in range(N):
    kept = []
    for prompt in prompts:
        completions = sample(policy, prompt, G, temperature=1.0)
        best = max(completions, key=reward)    # per-prompt best-of-G (our simplification; a tie: the first)
        kept.append((prompt, best))            # every prompt contributes, always
    policy = sft(policy, kept)
```

- The **outer loop is identical**; only the filter inside it differs
- **Self-distillation**: a constant bar, so a prompt contributes only if something clears it
- **RAFT**: a rank within the prompt. The weight depends on the other completions of the same prompt
  - so the one-step identity above is not derived for RAFT (skipped aside "conditions on the equivalence")
- *Next: what do the two filters keep, and what do they throw away?*


## Comparison: the two filters on the same twelve completions

> **A constant bar keeps what clears it; best-of-$G$ keeps one per prompt; GRPO (shown for contrast) gives signed weights.**

<img src="images/filter_shapes.png" alt="same twelve completions: a constant threshold, a per-prompt best-of-G, and GRPO's signed weights for contrast" width=36%>

- With the bar at 0.75, only prompt A's completions clear it. RAFT still keeps one completion per prompt, even prompt C's poor 0.41
- *Next: what do these filters keep, and what do they drop?*

## Comparison: what the filters keep, and what they drop

> **Keep this: both filters give a rejected completion weight 0. RLLM-10 asks: can a rejected completion, set against a kept one, teach?**
<!-- seed-plant: weight-zero-rejected -->

| Keeps | Drops (what each costs is our reading, not measured) |
|---|---|
| the model's **own completions** | any **explicit push down** on a bad completion |
| a **reward** on each | **magnitude**: 0.81 and 0.95 both clear a 0.75 bar |
| a way to **prefer better** | a **baseline** (PPO's value, RLLM-5; GRPO's group mean, RLLM-6) |

- **Why weight 0, not negative**: a negative weight says "worse than expected", and this loop has no baseline to say what to expect (RLLM-5)
  - the kept set is the whole signal; a rejected completion just leaves the batch
- A baseline constant within a prompt changes variance, not the expected direction (skipped aside "conditions on the equivalence")
  - GRPO's std and length division do change who gets weight
  - negative weights: left open (Think about 4)
- *Next: where do these methods sit on the three axes?*


## Placement: a hybrid, and the placing rule

> **A hybrid is a method where a reward picks the data and a reference trains on it.**
>
> **Placing rule: ask who supplies the weights. A reward judges the whole completion; a reference gives the token to imitate. A hybrid has both.**

- **Why this section?** These two use both a reward (the filter) and a reference (the SFT loss), so one column cannot hold them
- **Axis 1, whose samples**: the model's own, on-policy (RLLM-7), with some **staleness**
  - `sft` takes several steps on one batch: later steps use older parameters' completions
- **Axis 2, who supplies the weights**: a reward decides which completions count (1 or 0)
  - the kept tokens are the reference the SFT loss imitates
  - no negative weights, no baseline
- **Axis 3, how fine**: one per completion (0 or 1), used only to keep or reject
  - the update is per token: weight 1 on each token of a kept completion
- *Next: the map, with the hybrid box added*


## Placement: the map grows with the hybrid box

> **The hybrid box spans both columns in the own-samples row: a reward picks the data, a reference trains on it.**

<img src="images/second_map.png" alt="the first map with the hybrid box for self-distillation and RAFT filled and highlighted, spanning both columns of the own-samples row; the DPO and OPD slots still dashed" width=46%>

- **New in this session**: the hybrid box only. SFT, classical distillation, PPO and GRPO stay as in RLLM-8
- **Not in the reference column alone**: which tokens get weight 1 depends on the reward
- **Not in the reward column alone**: no negative weights, and no baseline subtracted
- The placements are **our reading** of how each method is built, not a result
- *Next: questions this loop leaves open*


## Think about

*The loop leaves open what a filter does when rewards are nearly all equal, and what repeated rounds do. State a prediction for each; these are questions, not claims.*

1. **Predict first.** Method Y: *sample $G$ completions per prompt, keep the top 25% by reward within each prompt, SFT on them.*
   - which of this session's two filters is it closer to?
   - place it with the three axes
2. **Predict first.** Raise the self-distillation bar from 0.75 to 0.99
   - what happens to the size of the kept set, and to the noise of the update?
3. Both methods put weight 0 on rejected completions
   - could you use them **and** keep an SFT-style loss?
   - what would a pair (a *better* and a *worse* completion for the same prompt) let you say that a single kept completion cannot?
4. Design a small experiment to test whether **negative weights matter in practice**
   - what would you hold fixed, and what would you measure?
5. **Predict first.** A prompt where **every** completion scores 0: what does each filter keep?
   - what does that say about how each filter treats **hard prompts**?


## What we established, and what it sets up

- **Established**: a filter is a **weight chosen by a reward**, so it breaks the zero push. A constant threshold gives a policy-gradient step
- **Limit of this session**: rejected completions get **weight 0**, so only kept ones teach. The RL equivalence covers **one expected step**
- **Sets up**: RLLM-10, Preference pairs and DPO
  - can the **rejected** completions teach? A **preference pair** keeps both a better and a worse completion


## Aside 1, justification: why a constant-threshold filter is a policy-gradient step

*For your own review; not worked through in session. Our derivation from the standard identity; the conditions are in aside 2.*

1. Let $\kappa$ be the filter's threshold and $R(\y)=1$ if $\rew(\y)\ge\kappa$, else 0. The objective is $J(\theta)=\Exp{\y\sim\pi_\theta}[R(\y)]$: the probability that a sampled completion clears the bar
2. The policy-gradient identity (RLLM-5): $\nabla J=\Exp{\y\sim\pi_\theta}[R(\y)\,\nabla_\theta\log\pi_\theta(\y)]$
3. The SFT loss on the kept set is $-\sum R(\y)\log\pi_\theta(\y)$ over $n$ samples
   - in expectation its gradient is $-n\,\nabla J$: **the same expression**
- The figure's toy checks it: the exact average push of the filter equals the numerical gradient of $J$ (asserted in the figure script)
- The reward is not in the loss; it enters the gradient only as the 0/1 multiplier $R$


## Aside 2, conditions on the equivalence

*For your own review; it qualifies aside 1.*

**Result, in words.** Descending the SFT loss on the kept set is, **in expectation**, a step of **policy gradient on a 0/1 reward with no baseline**, even though the reward enters *only* through the filter. It is *not* "filter-then-SFT is RL". It holds only under these conditions ($\kappa$ is the `threshold` in the pseudocode):

- **Expected direction, one step.** A round takes *many* SFT steps on one fixed batch
  - after the first step the batch is no longer on-policy, and nothing corrects for that (PPO's ratio and clipping, RLLM-5, exist for this)
  - the next round's *fresh* completions are on-policy again
- **A thresholded objective.** $J$ is the probability of clearing $\kappa$, not expected reward: 0.81 and 0.95 count the same, and everything below $\kappa$ is invisible
- **The constant-threshold filter only.** RAFT's best-of-$G$ is a *rank*-based weight, depending on the other completions of its prompt
  - the identity above was not derived for it (our reading)
- **A baseline that is constant within a prompt** (not depending on the completion's own value) leaves the expected direction unchanged and changes the variance
  - GRPO's group mean is *approximately* such a baseline
  - its division by the standard deviation, and by completion length, are separate choices that *do* change who gets weight
- **Sum vs. mean.** Averaging the SFT loss over each prompt's kept completions, instead of summing, keeps each prompt's direction
  - but prompts are re-weighted (if the number kept is treated as fixed)
- **No baseline** is why every weight is $\ge 0$; GRPO's group mean is where its negative weights come from


## Aside 3, sources: expert iteration and STaR

*For your own review.*

- **Expert iteration** (Anthony et al., 2017): an "expert" (there, a tree search) improves on the current policy, and a fast "apprentice" imitates it; repeat
- **STaR** (Zelikman et al., 2022): the model writes step-by-step reasoning; keep only reasoning that ends in the correct answer; fine-tune on it; repeat
  - STaR also hints the answer to the model on failed prompts; we skip that
- Both are close to sample, filter, imitate. The details differ, for example whether a round continues from the current policy or restarts from the base model
  - a fixed correct-or-not reward plays the constant threshold (**our reading**)
- **RAFT** (Dong et al., 2023) is the best-of-$G$ variant in the session


## Aside 4, the same loop in the DeepSeekMath table

*For your own review; we will not work through it in session.*

- **Source**: Shao et al. (2024), the DeepSeekMath paper, section 5.2.1 ("Towards to a Unified Paradigm"), with a table of methods
- Their gradient, up to notation (and a $1/|o|$ **length normalization** that we leave out):
  $$\nabla_\theta J_{\mathcal A}(\theta)=\Exp{(q,o)\sim\mathcal D}\Big[\frac{1}{|o|}\sum_t GC_{\mathcal A}(q,o,t)\,\nabla_\theta\log\pi_\theta(o_t\mid q,o_{<t})\Big]$$
  - $GC$ is the **gradient coefficient**: our weight $w_\tt$
  - the **data source** $\mathcal D$ is a **separate component**: our whose-samples axis from RLLM-7
- They name **three components** that differ between methods: the data source, the algorithm (how $GC$ is computed), and the reward signal
- Their table, in our words:

| Method | Data source | Gradient coefficient $GC$ |
|---|---|---|
| SFT | human-written demonstrations | $1$ |
| RFT | completions from the **SFT model** | $1$ if the answer is correct, else $0$ |
| Online RFT | completions from the **current** model | the same $0/1$ |
| DPO | pairs of completions | $\sigma(-z)$-type weight: **up** on the winner's tokens, **down** on the loser's |
| PPO | completions from the current model | the advantage $\hat{\advseq}_\tt$ |
| GRPO | groups of completions from the current model | the group-relative advantage, plus a KL term |

- **"RFT" in this table is the paper's name for rejection-sampling fine-tuning**: sample, keep the correct answers, then SFT on them
  - it is **not** this series' "reinforcement fine-tuning" (RLLM-1)
- **Reading it**: RFT and Online RFT differ only in the **data** column
  - Online RFT is this session's loop with a correctness filter
  - SFT and RFT differ in the **coefficient**. That is our two axes
- Their paper uses this to **compare methods**, as we do. The details of each coefficient are in its appendix


## Aside 5, names that clash, and a trap in RAFT

*For your own review; not worked through in session.*

- **Self-distillation** here means the loop with a constant-bar filter. It is **not** self-critique (RLLM-3), and papers also use the word for one model taught with extra context: that is RLLM-12's setting, not this loop
- **RFT**: DeepSeekMath's "RFT" means rejection-sampling fine-tuning (this loop with a correctness filter). This series' RFT is reinforcement fine-tuning (RLLM-1)
- **A trap in RAFT**: a group in which every completion scores 0 still keeps one completion (the first of the tie), so RAFT trains on a failure


## Reading for this session

- **Suggested**: for a paper, at least the **abstract and introduction**. For other items, the parts named
  - [Zelikman, Wu, Mu and Goodman (2022), *STaR: Bootstrapping Reasoning With Reasoning*](https://arxiv.org/abs/2203.14465): keep what works, imitate
  - [Dong et al. (2023), *RAFT: Reward rAnked FineTuning for Generative Foundation Model Alignment*](https://arxiv.org/abs/2304.06767): ranked completions as SFT data
- **Additional** (4 more, for digging deeper), and the other sources this session cites: `reading_list.ipynb`
- Links were checked in October 2026; tell us if one has moved
